# 23 — SFT, DPO, RLHF, and GRPO

In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Pretraining learns language. Post-training shapes behavior.

- **SFT:** imitate high-quality instruction responses.
- **RLHF:** train a reward model from preferences, then optimize policy.
- **DPO:** optimize directly on chosen vs rejected pairs.
- **GRPO:** use groups of sampled completions and relative rewards; often discussed for reasoning-style training.

This notebook implements toy losses. Production post-training needs careful safety, evals, and data governance.

In [ ]:
# SFT loss: teacher-forced next-token cross entropy.
B,T,V = 2,5,10
logits = torch.randn(B,T,V, requires_grad=True)
labels = torch.randint(0,V,(B,T))
sft_loss = F.cross_entropy(logits.reshape(-1,V), labels.reshape(-1))
print('SFT loss:', float(sft_loss))


In [ ]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):
    # inputs are sequence log probabilities under policy/reference.
    pi_logratios = policy_chosen - policy_rejected
    ref_logratios = ref_chosen - ref_rejected
    logits = beta * (pi_logratios - ref_logratios)
    return -F.logsigmoid(logits).mean()

policy_chosen = torch.tensor([-5.0, -3.0])
policy_rejected = torch.tensor([-6.0, -4.5])
ref_chosen = torch.tensor([-5.5, -3.2])
ref_rejected = torch.tensor([-5.7, -4.0])
print('DPO loss:', float(dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)))


In [ ]:
class RewardModel(nn.Module):
    def __init__(self, d=8):
        super().__init__()
        self.score = nn.Linear(d, 1)
    def forward(self, x):
        return self.score(x).squeeze(-1)

rm = RewardModel()
chosen_emb = torch.randn(4,8)
rejected_emb = torch.randn(4,8)
reward_chosen = rm(chosen_emb)
reward_rejected = rm(rejected_emb)
reward_loss = -F.logsigmoid(reward_chosen - reward_rejected).mean()
print('reward model pairwise loss:', float(reward_loss))


In [ ]:
def grpo_group_advantages(rewards, eps=1e-8):
    # rewards: [prompts, completions_per_prompt]
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True).clamp_min(eps)
    return (rewards - mean) / std

rewards = torch.tensor([[0.2, 0.5, 0.1, 0.9], [1.0, 0.7, 0.6, 0.65]])
advantages = grpo_group_advantages(rewards)
print(advantages)


## Production templates

```python
# DPO with TRL style APIs, when installed:
# from trl import DPOTrainer, DPOConfig
# trainer = DPOTrainer(model=model, ref_model=ref_model, args=DPOConfig(...), train_dataset=dataset)
# trainer.train()

# GRPO with TRL style APIs, when installed:
# from trl import GRPOTrainer, GRPOConfig
# trainer = GRPOTrainer(model=model, reward_funcs=[my_reward], args=GRPOConfig(...), train_dataset=dataset)
# trainer.train()
```

## Founder notes

Post-training is product design as much as ML. Your preference data encodes tone, brand, risk tolerance, refusal behavior, helpfulness, and escalation policy. Bad preference data creates bad product behavior.